# TensorFlow Fundus Image Classification — Two-Stage Training

This notebook trains **4 TensorFlow models** in a two-stage pipeline:

**Stage 1 — Binary:** Healthy vs Unhealthy  
**Stage 2 — Disease:** Classify the specific disease (5 major conditions)

**Models:**
- DenseNet121 (2017)
- ConvNeXtTiny (2022)
- MobileNetV2 (2018)
- ResNet18 (Custom, from scratch)

**Disease Classes:** AMD, Cataract, Diabetic Retinopathy, Glaucoma, Myopia

**Run this on Google Colab with GPU enabled.**

In [ ]:
import os
import json
import time
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    DenseNet121,
    ConvNeXtTiny,
    MobileNetV2,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

# All 26 original folder names in the dataset
ALL_CLASS_NAMES = [
    "AMD", "Bleeding", "Blur Fundus", "Cataract", "Coats", "Cotton Wool Spots",
    "Diabetic Retinopathy", "Drusen", "Glaucoma", "Hemorrhage",
    "Healthy", "Hypertensive Retinopathy", "Laser Scars", "Macular Hole",
    "Maculopathy", "Myopia", "Normal Fundus", "Optic Disc Pallor",
    "Preretinal Hemorrhage", "Proliferative DR", "Retinal Detachment",
    "Retinitis Pigmentosa", "Retinoblastoma", "STARE Normal",
    "Toxoplasmosis", "Vessel Tortuosity",
]

# Classes that count as "Healthy"
HEALTHY_CLASSES = {"Healthy", "Normal Fundus", "STARE Normal"}

# Stage 2: Only the 5 major diseases (focused for better accuracy)
DISEASE_NAMES = ["AMD", "Cataract", "Diabetic Retinopathy", "Glaucoma", "Myopia"]
NUM_DISEASES = len(DISEASE_NAMES)

IMG_SIZE = 224  # Match ImageNet pretraining resolution
BATCH_SIZE = 32
EPOCHS = 20

print(f"Healthy classes: {HEALTHY_CLASSES}")
print(f"Disease classes ({NUM_DISEASES}): {DISEASE_NAMES}")

## Upload / Mount Your Dataset

Update `DATASET_ROOT` to point to your fundus image dataset.  
Expected structure:
```
dataset/
  train/
    AMD/
    Bleeding/
    ...
    Healthy/
    Normal Fundus/
    ...
  test/
    AMD/
    ...
```

In [ ]:
# Mount Google Drive (uncomment if your dataset is on Drive)
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_ROOT = "./dataset"  # Update this path
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

assert os.path.isdir(TRAIN_DIR), f"Train directory not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Test directory not found: {TEST_DIR}"

In [ ]:
# ── Reorganize dataset into binary + disease folders ─────────────────────
# Creates symlink-based directory structures so we don't duplicate data.

BINARY_DIR = "./dataset_binary"
DISEASE_DIR = "./dataset_disease"


def make_binary_and_disease_dirs(src_split, split_name):
    """Create binary (Healthy/Unhealthy) and disease-only directory structures."""
    # Binary dirs
    for label in ["Healthy", "Unhealthy"]:
        os.makedirs(os.path.join(BINARY_DIR, split_name, label), exist_ok=True)

    # Disease dirs (only the 5 major diseases)
    for cls in DISEASE_NAMES:
        os.makedirs(os.path.join(DISEASE_DIR, split_name, cls), exist_ok=True)

    for cls in ALL_CLASS_NAMES:
        src_cls_dir = os.path.join(src_split, cls)
        if not os.path.isdir(src_cls_dir):
            continue

        is_healthy = cls in HEALTHY_CLASSES
        binary_label = "Healthy" if is_healthy else "Unhealthy"

        for fname in os.listdir(src_cls_dir):
            src_path = os.path.abspath(os.path.join(src_cls_dir, fname))

            # Binary: copy into Healthy or Unhealthy
            binary_dst = os.path.join(
                BINARY_DIR, split_name, binary_label, f"{cls}_{fname}"
            )
            if not os.path.exists(binary_dst):
                os.symlink(src_path, binary_dst)

            # Disease: only the 5 major disease classes
            if cls in DISEASE_NAMES:
                disease_dst = os.path.join(DISEASE_DIR, split_name, cls, fname)
                if not os.path.exists(disease_dst):
                    os.symlink(src_path, disease_dst)


# Clean up previous runs
for d in [BINARY_DIR, DISEASE_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)

make_binary_and_disease_dirs(TRAIN_DIR, "train")
make_binary_and_disease_dirs(TEST_DIR, "test")

# Count images
for label in ["Healthy", "Unhealthy"]:
    n_train = len(os.listdir(os.path.join(BINARY_DIR, "train", label)))
    n_test = len(os.listdir(os.path.join(BINARY_DIR, "test", label)))
    print(f"Binary — {label}: {n_train} train, {n_test} test")

print(f"\nDisease classes ({NUM_DISEASES}):")
for cls in DISEASE_NAMES:
    train_dir = os.path.join(DISEASE_DIR, "train", cls)
    test_dir = os.path.join(DISEASE_DIR, "test", cls)
    n_train = len(os.listdir(train_dir)) if os.path.isdir(train_dir) else 0
    n_test = len(os.listdir(test_dir)) if os.path.isdir(test_dir) else 0
    print(f"  {cls}: {n_train} train, {n_test} test")

In [ ]:
# ── Data generators ────────────────────────────────────────────────────────

train_aug = ImageDataGenerator(
    rescale=1.0 / 255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    validation_split=0.15,
)

test_aug = ImageDataGenerator(rescale=1.0 / 255)


def make_generators(data_dir, class_names):
    """Create train/val/test generators for a given directory and class list."""
    train_gen = train_aug.flow_from_directory(
        os.path.join(data_dir, "train"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=class_names,
        subset="training",
        shuffle=True,
    )
    val_gen = train_aug.flow_from_directory(
        os.path.join(data_dir, "train"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=class_names,
        subset="validation",
        shuffle=False,
    )
    test_gen = test_aug.flow_from_directory(
        os.path.join(data_dir, "test"),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        classes=class_names,
        shuffle=False,
    )
    return train_gen, val_gen, test_gen


# Stage 1: Binary generators
print("=== Stage 1: Binary (Healthy vs Unhealthy) ===")
binary_train, binary_val, binary_test = make_generators(
    BINARY_DIR, ["Healthy", "Unhealthy"]
)

# Stage 2: Disease generators
print("\n=== Stage 2: Disease Classification ===")
disease_train, disease_val, disease_test = make_generators(
    DISEASE_DIR, DISEASE_NAMES
)

In [ ]:
# ── Custom ResNet18 (from scratch) ────────────────────────────────────────

def _residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


def build_custom_resnet18(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
    for filters, stride in [(64, 1), (64, 1)]:
        x = _residual_block(x, 64, stride)
    for i, stride in enumerate([2, 1]):
        x = _residual_block(x, 128, stride)
    for i, stride in enumerate([2, 1]):
        x = _residual_block(x, 256, stride)
    for i, stride in enumerate([2, 1]):
        x = _residual_block(x, 512, stride)
    x = layers.GlobalAveragePooling2D()(x)
    return tf.keras.Model(inputs, x, name="custom_resnet18")


print("ResNet18 builder ready.")

In [ ]:
# ── Model builder ─────────────────────────────────────────────────────────

MODELS = {
    "DenseNet121":       {"builder": DenseNet121,   "custom": False},
    "ConvNeXtTiny":      {"builder": ConvNeXtTiny,  "custom": False},
    "MobileNetV2":       {"builder": MobileNetV2,   "custom": False},
    "ResNet18 (Custom)": {"builder": None,          "custom": True},
}


def build_model(name, info, num_classes):
    """Build a classifier with the given backbone and output size."""
    input_shape = (IMG_SIZE, IMG_SIZE, 3)

    if info["custom"]:
        base = build_custom_resnet18(input_shape)
        head = []
    else:
        base = info["builder"](
            weights="imagenet",
            include_top=False,
            input_shape=input_shape,
        )
        base.trainable = True
        total_layers = len(base.layers)
        freeze_until = int(total_layers * 0.7)
        for layer in base.layers[:freeze_until]:
            layer.trainable = False
        head = [layers.GlobalAveragePooling2D()]

    activation = "sigmoid" if num_classes == 2 else "softmax"
    out_units = 1 if num_classes == 2 else num_classes

    model = models.Sequential([
        base,
        *head,
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(out_units, activation=activation),
    ])

    loss = "binary_crossentropy" if num_classes == 2 else "categorical_crossentropy"

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=loss,
        metrics=["accuracy"],
    )
    return model


print("Model builder ready.")

In [ ]:
# ── Training helper ───────────────────────────────────────────────────────

def train_and_evaluate(stage_name, model_registry, train_gen, val_gen, test_gen, num_classes, weight_prefix):
    """Train all models for a given stage and return results dict."""
    os.makedirs("weights", exist_ok=True)
    results = {}

    for model_name, info in model_registry.items():
        print(f"\n{'='*60}")
        print(f"[{stage_name}] Training: {model_name}")
        print(f"{'='*60}")

        model = build_model(model_name, info, num_classes)
        model.summary()

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=5, restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
            ),
        ]

        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=callbacks,
        )

        # Evaluate
        print(f"\nEvaluating {model_name} on test set...")
        test_loss, test_acc = model.evaluate(test_gen)

        # Inference timing
        sample_batch = next(iter(test_gen))[0][:1]
        model.predict(sample_batch, verbose=0)  # warm-up
        start = time.perf_counter()
        for _ in range(50):
            model.predict(sample_batch, verbose=0)
        inf_ms = (time.perf_counter() - start) / 50 * 1000

        results[model_name] = {
            "test_accuracy": float(test_acc),
            "test_loss": float(test_loss),
            "inference_ms": float(inf_ms),
            "best_val_accuracy": float(max(history.history["val_accuracy"])),
            "epochs_trained": len(history.history["loss"]),
        }

        # Save weights
        safe_name = model_name.replace(" ", "_").replace("(", "").replace(")", "")
        weight_path = f"weights/{weight_prefix}_{safe_name}.keras"
        model.save(weight_path)
        print(f"  Saved: {weight_path}")
        print(f"  Test Accuracy: {test_acc:.4f}")
        print(f"  Inference:     {inf_ms:.1f} ms/image")

        del model
        tf.keras.backend.clear_session()

    return results

In [ ]:
# ── Stage 1: Binary (Healthy vs Unhealthy) ───────────────────────────────

print("STAGE 1: Binary Classification — Healthy vs Unhealthy")
print("=" * 60)

binary_results = train_and_evaluate(
    stage_name="Binary",
    model_registry=MODELS,
    train_gen=binary_train,
    val_gen=binary_val,
    test_gen=binary_test,
    num_classes=2,
    weight_prefix="binary",
)

print("\nStage 1 complete!")
for name, res in binary_results.items():
    print(f"  {name}: {res['test_accuracy']:.2%}")

In [ ]:
# ── Stage 2: Disease Classification (5 major diseases) ───────────────────

print("STAGE 2: Disease Classification — 5 major diseases")
print("=" * 60)

disease_results = train_and_evaluate(
    stage_name="Disease",
    model_registry=MODELS,
    train_gen=disease_train,
    val_gen=disease_val,
    test_gen=disease_test,
    num_classes=NUM_DISEASES,
    weight_prefix="disease",
)

print("\nStage 2 complete!")
for name, res in disease_results.items():
    print(f"  {name}: {res['test_accuracy']:.2%}")

In [ ]:
# ── Export results ────────────────────────────────────────────────────────

all_results = {
    "binary": binary_results,
    "disease": disease_results,
    "config": {
        "img_size": IMG_SIZE,
        "healthy_classes": sorted(HEALTHY_CLASSES),
        "disease_names": DISEASE_NAMES,
    },
}

with open("weights/tf_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

print("Results saved to weights/tf_results.json")
print(json.dumps(all_results, indent=2))

In [ ]:
# ── Download weights ──────────────────────────────────────────────────────

shutil.make_archive("tf_weights", "zip", ".", "weights")

try:
    from google.colab import files
    files.download("tf_weights.zip")
    print("Download triggered!")
except ImportError:
    print("Not on Colab — copy the weights/ folder manually.")

print("\nWeight files:")
for f in sorted(os.listdir("weights")):
    size_mb = os.path.getsize(os.path.join("weights", f)) / 1e6
    print(f"  weights/{f}  ({size_mb:.1f} MB)")

## Next Steps

1. Download `tf_weights.zip` from the output above
2. Extract the `weights/` folder into your `intelligenteye/` project root
3. Run `streamlit run app.py`

The app will:
- **Stage 1:** Classify the image as Healthy or Unhealthy
- **Stage 2:** If Unhealthy, predict the specific disease